# Домашняя работа 2 — качество данных

## Подготовка

In [1]:
from pathlib import Path
import pandas as pd
from pandas.testing import assert_frame_equal
DATA = Path("../data")
LOCAL = Path("../data")
WORK = Path("../results")
WORK.mkdir(exist_ok=True)
print("pandas", pd.__version__)

pandas 2.2.2


## Независимая загрузка

In [2]:
decisions = pd.read_csv(
    DATA / "decisions.csv", dtype={"lei": "string", "county_code": "string"}
)
export = pd.read_csv(LOCAL / "training_export.csv", dtype="string")
raw_copy = export.copy(deep=True)
print(decisions.shape, export.shape)
print(export.dtypes)
export.head(5)

(6830, 7) (6832, 7)
source_row       string[python]
activity_year    string[python]
lei              string[python]
county_code      string[python]
action_taken     string[python]
loan_amount      string[python]
income           string[python]
dtype: object


,source_row,activity_year,lei,county_code,action_taken,loan_amount,income
0,2,2023,549300VZVN841I2ILS84,11001,1,oops,72.0
1,3,2023,549300P1347BFCBHJV60,11001,88,585000.0,174.0
2,5,2023,549300P1347BFCBHJV60,11001,1,275000.0,BAD
3,7,2023,549300RPOGWJRH63HS39,11001,1,295000.0,63.0
4,8,2023,549300RPOGWJRH63HS39,11001,1,1125000.0,247.0


## H1. Собственная функция очистки

In [3]:
def clean_export(source):
    df = source.copy(deep=True)
    has_id = df["source_row"].notna()
    unique_check = df.loc[has_id].drop_duplicates()
    assert unique_check["source_row"].is_unique, "Найдены конфликтующие версии source_row"
    dup_mask = has_id & df["source_row"].duplicated()
    replays_count = int(dup_mask.sum())
    df = df[~dup_mask].copy()
    df["county_code"] = df["county_code"].str.strip()
    parse_errors = {}
    numeric_cols = ["source_row", "loan_amount", "income", "action_taken"]
    for col in numeric_cols:
        converted = pd.to_numeric(df[col], errors="coerce")
        errors = (df[col].notna() & converted.isna()).sum()
        parse_errors[col] = int(errors)
        df[col] = converted
    bad_id = df["source_row"].isna()
    bad_action = ~df["action_taken"].isin([1, 2, 3])
    bad_loan = df["loan_amount"].isna() | (df["loan_amount"] <= 0)
    bad_county = ~df["county_code"].eq("11001").fillna(False)
    reason = pd.Series("", index=df.index, dtype="string")
    reason[bad_id] += "unknown_source_row;"
    reason[bad_action] += "invalid_action_taken;"
    reason[bad_loan] += "invalid_loan_amount;"
    reason[bad_county] += "invalid_county;"
    invalid_mask = reason != ""
    clean_data = df[~invalid_mask].copy()
    quarantine_data = source.loc[df.index[invalid_mask]].copy()
    quarantine_data["reason"] = reason[invalid_mask]
    audit_log = {
        "input": len(source),
        "replays": replays_count,
        "clean": len(clean_data),
        "quarantine": len(quarantine_data),
        "parse_errors": parse_errors
    }
    assert audit_log["input"] == audit_log["replays"] + audit_log["clean"] + audit_log["quarantine"], "Баланс не сходится!"
    return clean_data, quarantine_data, audit_log
result, rejected, audit = clean_export(export)

## H2. Год и известный доход

In [4]:
year_ok = result["activity_year"].eq("2023").fillna(False).all()
assert year_ok, "Не все activity_year равны '2023'"
print("Проверка года: Все activity_year в result равны '2023'.")
total_rows = len(result)
known_income = result["income"].notna().sum()
unknown_income = result["income"].isna().sum()
mean_income = result["income"].mean()
print(f"Общее количество строк в result: {total_rows}")
print(f"Количество известных income: {known_income}")
print(f"Количество неизвестных income: {unknown_income}")
print(f"Средний известный доход: {mean_income:.2f} (тысяч долларов)")
assert audit["input"] == audit["replays"] + audit["clean"] + audit["quarantine"]
print("Баланс и сохранность входа подтверждены.\n")

Проверка года: Все activity_year в result равны '2023'.
Общее количество строк в result: 6828
Количество известных income: 6485
Количество неизвестных income: 343
Средний известный доход: 237.07 (тысяч долларов)
Баланс и сохранность входа подтверждены.



## H3. Проверка отчётного года

In [5]:
wrong_year = export.copy(deep=True)
wrong_year.loc[wrong_year.index[0], "activity_year"] = "2022"
year_ok_wrong = wrong_year["activity_year"].eq("2023").fillna(False).all()
print(f"Проверка wrong_year обнаружила нарушение (должно быть True): {not year_ok_wrong}")
assert not year_ok_wrong, "Проверка не сработала!В"
pd.testing.assert_frame_equal(export, raw_copy)
print("Проверка: исходный export полностью сохранился и не был изменен.")

Проверка wrong_year обнаружила нарушение (должно быть True): True
Проверка: исходный export полностью сохранился и не был изменен.


## H+1. Конфликт исходных значений

## H+2. Совпадение опубликованных полей